In [20]:
using LinearAlgebra
using BenchmarkTools

# Precomputes standard method:
Given a the monomial coefficients of a polynomial $b_0,\ldots,b_8$ we have

$$
\begin{align}
    f_4 = b_8\\
    c_1 = b_7/(2f_4)\\
    t_2 = b_6/f_4 - c_1^2\\
    t_1 = b_5/f_4 - c_1t_2\\
    d_0 = \frac{1}{4} ( 1 - t_2^2 + 4b_4/f_4 - 4c_1t_1)\\
    e_2 = \frac{1}{2} ( t_2 + 1 )\\
    d_2 = t_2 - e_2 =  \frac{1}{2} ( t_2 - 1 )\\
    e_1 = c_1d_0 + t_1e_2 - b_3/f_4\\
    f_2  = b_2 - f_4(d_0e_2 + d_1e_1)  \\
    f_1  = b_1 - f_4d_0e_1  \\
    f_0  = b_0  \\
\end{align}
$$
Plus the equation
$$
    d_1 = t_1 - e_1
$$

Computed below

In [21]:
# Graph formulation of any polynomial of deg 8
function p8_coeff_precompute(b)
    #b₀, b₁, b₂, b₃, b₄, b₅, b₆, b₇, b₈ = mc
    f₄ = b[8+1]
    c₁ = b[7+1] / (2*f₄)
    t₂ = b[6+1]/f₄ - c₁^2
    t₁ = b[5+1]/f₄ - c₁*t₂
    d₀ = 1/4 * (1 - t₂^2 + 4*b[4+1]/f₄ - 4*c₁*t₁)

    e₂ = 1/2 * (t₂ + 1)
    d₂ = t₂ - e₂

    e₁ = c₁*d₀ + t₁*e₂ - b[3+1]/f₄
    d₁ = t₁ - e₁

    f₂ = b[2+1] - f₄*(d₀*e₂ + d₁*e₁)
    f₁ = b[1+1] - f₄*d₀*e₁
    f₀ = b[0+1]
    return f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁
end

p8_coeff_precompute (generic function with 1 method)

Note that 1+d₂ can be used in place of e₂ when evaluating

For the memory efficient variant, we additionally compute
$$
\begin{align}
          r_1 &\coloneqq d_1 - \frac{1}{2}c_1(d_2 - \frac{1}{4}c_1^2), \\
          r_2 &\coloneqq d_2 - \frac{1}{4}c_1^2,    \\
          r_3 &\coloneqq e_1 - d_1 - \frac{1}{2} c_1,  \\  
          r_4 &\coloneqq f_1 + f_2(e_1-d_1).
\end{align}
$$
below

In [22]:
# Additional memory efficient precomputes
function p8_mem3_precompute(b)
    f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁ = p8_coeff_precompute(b)
    r₁ = d₁ - 1/2 * c₁ *(d₂ - 1/4 * c₁^2)
    r₂ = d₂ - 0.25* c₁^2
    r₃ = e₁ - d₁ - 0.5 * c₁
    r₄ = f₁ - f₂*(e₁- d₁)
    return f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁, r₁, r₂, r₃, r₄
end

p8_mem3_precompute (generic function with 1 method)

# Base Variant
### The following function computes any degree-8 polynomial using only 3 matrix-matrix multiplications

In [23]:
# memory inconsiderate version:
function sp8_eval(S₁, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁)
    S₀ = I
    #S₁ = X
    S₂ = S₁*S₁
    S₃ = S₂*(c₁ *S₁ + S₂)
    S₄ = (d₀ *S₀ + d₁ * S₁ + d₂ *S₂ + S₃)*(e₁ *S₁ + (1+d₂)*S₂ + S₃)
    return f₀*S₀ + f₁*S₁ + f₂*S₂ + f₄*S₄
end

sp8_eval (generic function with 1 method)

# Memory Efficient Variant
### The following function constructs the output matrix using only 3 memory slots and 3 matrix-matrix multiplications

$$M_2 \leftarrow M_1 M_1 + \frac{1}{2}c_1M_1$$
$$M_3 \leftarrow M_2 M_2 + r_2 M_2$$
$$M_3 \leftarrow M_3 + r_1 M_1$$
$$M_2 \leftarrow M_2 + r_3 M_1$$
$$M_1 \leftarrow r_4 M_1 + f_2 M_2 + f_0 I$$
$$M_2 \leftarrow M_2 + M_3$$
$$M_3 \leftarrow M_3 + d_0 I$$
$$M_1 \leftarrow M_1 + f_4 M_2 M_3$$
$$M_1 \leftarrow p(M_1)$$

In [24]:
function p8_mem3!(M₁, M₂, M₃, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁, r₁, r₂, r₃, r₄)
    M₂ .= M₁
    mul!(M₂, M₁, M₁, 1.0, 0.5*c₁)
    mul!(M₃, M₂, M₂)
    BLAS.axpby!(r₂, M₂, 1.0, M₃)
    BLAS.axpby!(r₁, M₁, 1.0, M₃)
    BLAS.axpby!(r₃, M₁, 1.0, M₂)
    BLAS.axpby!(f₂, M₂, r₄, M₁)
    for i =1:size(M₁)[1]
        M₁[i,i] += f₀
    end;
    BLAS.axpy!(1.0,  M₃, M₂)
    for i =1:size(M₁)[1]
        M₃[i,i] += d₀
    end;
    mul!(M₁, M₂, M₃, f₄, 1.0)
end

p8_mem3! (generic function with 1 method)

## Precomputations and Workspace Setup

In [ ]:
#monomial coefficients
mc = randn(9)
f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁, r₁, r₂, r₃, r₄ = Float64.(p8_mem3_precompute(mc));
n = 5000
M₁ = randn(n,n)
M₂ = zeros(n,n) # More efficient for this to be a copy of X/M₁ when intialized
M₃ = zeros(n,n);

## Error comparison

In [ ]:
S₅ = sp8_eval(M₁, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁);
p8_mem3!(M₁, M₂, M₃, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁, r₁, r₂, r₃, r₄);
relerr = norm(S₅ - M₁)/(norm(S₅));
@show relerr

relerr = 1.3472866556010951e-15


1.3472866556010951e-15

## Timing and Allocation comparison

In [27]:
@btime sp8_eval(M₁, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁);

  492.387 ms (57 allocations: 579.84 MiB)


In [28]:
@btime p8_mem3!(M₁, M₂, M₃, f₀, f₁, f₂, f₄, d₀, d₁, d₂, c₁, e₁, r₁, r₂, r₃, r₄);

  433.395 ms (0 allocations: 0 bytes)
